In [8]:
import os, json, time

In [9]:
from together import Together
import utils

key_file = 'together-lab-key.txt'
with open(key_file, 'r') as f:
    API_KEY = f.read().strip()

client = Together(
  api_key=API_KEY
)


In [ ]:
from importlib import reload
reload(utils)

In [11]:
model_name = 'qwq'
model_endpoint = utils.model_names_to_endpoints[model_name]
model_endpoint

'Qwen/QwQ-32B'

In [12]:
data_dir = '../data/final_dataset'
short_ans_files = ['certamen_short_answer.json', 'junior_scholarship_short_answer.json']
short_ans_files = [os.path.join(data_dir, f) for f in short_ans_files]

file_to_data = {}
for file in short_ans_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

    print(base_name, len(file_to_data[base_name]))

certamen_short_answer.json 4596
junior_scholarship_short_answer.json 675


In [13]:
def construct_short_ans_one_word_user_prompt(q_dict):
    question_text = q_dict['question'] if 'question' in q_dict else q_dict['question_text']
    question_text += '\n' + utils.short_ans_one_word_format_instructions

    return question_text

In [9]:
prompt = construct_short_ans_one_word_user_prompt(file_to_data['certamen_short_answer.json'][0])
prompt

'What name was given to the large agricultural estates which resulted from the distribution of the ager publicus in the 2nd century B.C.?\nAt the end of your response, give your answer as a single word like this:\nAnswer: Word'

In [10]:
response = client.chat.completions.create(
  model=model_endpoint,
  messages=[
    {
        "role": "system",
        "content": utils.sys_prompt
    },
    {
      "role": "user",
      "content": prompt
    }
  ],
  temperature=0.6, 
  top_p=0.95, 
  #min_p=0,
  #top_k=20
)
print(response.choices[0].message.content)

<think>
Okay, let's see. The question is asking about the name given to large agricultural estates that came from the distribution of the ager publicus in the 2nd century BCE. Hmm, I need to recall my Roman history here.

First, I remember that ager publicus refers to public land owned by the Roman state. In the Republic period, there were issues with how this land was distributed. The Gracchi brothers, Tiberius and Gaius, tried to reform land distribution because the wealthy were accumulating too much of it, right? So the problem was that the rich, especially the senators and equites, were taking over large tracts of public land, often using slaves to work it instead of small farmers.

These large estates... what were they called? I think the term is "latifundia." Let me make sure. The latifundia were indeed those big estates, especially in places like Sicily and southern Italy, where they grew cash crops like grain or olives. The Gracchan land reforms aimed to limit how much land an 

In [6]:
save_dir = f'../data/model_responses/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

In [14]:
for filename, data in file_to_data.items():
    print(filename)
    q_id_to_resp = {}
    i = 0
    save_file = os.path.join(save_dir, filename)
    if os.path.exists(save_file):
        with open(save_file, 'r') as f:
            q_id_to_resp = json.load(f)
        
    for q_dict in data:
        q_id = q_dict['question_id']
        if q_id in q_id_to_resp:
            i += 1
            continue
        prompt = construct_short_ans_one_word_user_prompt(q_dict)

        response = client.chat.completions.create(
            model=model_endpoint,
            messages=[
                {
                    "role": "system",
                    "content": utils.sys_prompt
                },
                {
                "role": "user",
                "content": prompt
                }
            ],
            temperature=0.6, 
            top_p=0.95, 
            #min_p=0,
            #top_k=20
        )
        try:
            resp = response.choices[0].message.content
        except:
            print(f'Error on {q_id}')
            print(response)
            # dump this file 
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            break
        q_id_to_resp[q_id] = resp

        time.sleep(.1)
        

        if i % 50 == 0:
            # dump this file 
            print(f'  {i} / {len(data)}')
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            
        i += 1

    # dump this file 
    with open(save_file, 'w') as f:
        json.dump(q_id_to_resp, f, indent=4)

certamen_short_answer.json
  3400 / 4596
  3500 / 4596
  3600 / 4596
  3700 / 4596
  3800 / 4596
  3900 / 4596
  4000 / 4596
  4100 / 4596
  4200 / 4596


ServiceUnavailableError: Error code: 503 - The server is overloaded or not ready yet.